# Exploratory Data Analysis

This notebook explores synthetic daily hospital capacity data.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
np.random.seed(42)

In [3]:
n_days = 200

data = pd.DataFrame({
    "day": range(1, n_days + 1),
    "admissions": np.random.poisson(50, n_days),
    "avg_length_of_stay": np.random.randint(3, 7, n_days),
    "available_beds": 160
})

In [4]:
occupancy = []

for t in range(len(data)):
    beds = 0
    for past_day in range(t + 1):
        days_since = t - past_day
        if days_since < data.loc[past_day, "avg_length_of_stay"]:
            beds += data.loc[past_day, "admissions"]
    occupancy.append(beds)

data["occupancy"] = occupancy

In [5]:
data["capacity_stress"] = (
    data["occupancy"] > data["available_beds"]
).astype(int)

In [6]:

data


,day,admissions,avg_length_of_stay,available_beds,occupancy,capacity_stress
0,1,47,4,160,47,0
1,2,55,6,160,102,0
2,3,42,4,160,144,0
3,4,52,6,160,196,1
4,5,58,4,160,207,1
...,...,...,...,...,...,...
195,196,49,6,160,253,1
196,197,52,6,160,191,1
197,198,33,6,160,224,1
198,199,49,6,160,273,1


In [15]:
X = data[["admissions", "avg_length_of_stay"]]
y = data["capacity_stress"]

In [16]:
from sklearn.model_selection import cross_val_score

model = LogisticRegression()

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="roc_auc"
)

print("AUC scores:", scores)
print("Mean AUC:", scores.mean())

AUC scores: [0.52285714 0.46857143 0.54513889 0.55555556 0.47222222]
Mean AUC: 0.5128690476190476


In [17]:
from sklearn.model_selection import cross_val_predict

y_prob_cv = cross_val_predict(
    LogisticRegression(),
    X,
    y,
    cv=5,
    method="predict_proba"
)[:,1]

print("Cross-validated AUC:", roc_auc_score(y, y_prob_cv))

Cross-validated AUC: 0.4925944841675179


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [9]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [10]:
y_pred = model.predict(X_test)

In [11]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         6
           1       0.90      1.00      0.95        54

    accuracy                           0.90        60
   macro avg       0.45      0.50      0.47        60
weighted avg       0.81      0.90      0.85        60



/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
y_prob = model.predict_proba(X_test)[:, 1]
print("ROC AUC:", roc_auc_score(y_test, y_prob))

ROC AUC: 0.4907407407407408


In [13]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

coefficients

,Feature,Coefficient
0,admissions,0.039142
1,avg_length_of_stay,-0.102101


In [14]:
import numpy as np

coefficients["Odds_Ratio"] = np.exp(coefficients["Coefficient"])
coefficients

,Feature,Coefficient,Odds_Ratio
0,admissions,0.039142,1.039919
1,avg_length_of_stay,-0.102101,0.902938


### Reflection

The model's poor ROC AUC indicates that admissions and LOS alone are insufficient predictors of stress. Since occupancy directly determines stress, excluding it significantly reduces predictive power. This demonstrates the trade-off between predictive accuracy and avoiding data leakage (since occupancy and stress are co-dependent variables).

Problems: 
- P(y=1)= 54/60=0.9, no need for predicting, saying 1 always already is 90% correct (log-loss minimiert)
- ROC - modell worse than random
- Logistic regressaion ist stateless, no memory, only day-to-day values
- accuracy = richtig klassifiziert(TP+TN)/gesamt fälle, global, does not differentiate between class ratio
- ROC checks probability stress (Recall/TP rate v FP rate) for different values for p: 0.1,0.2.0.3 etc.
- support of positive class: P=TP+FN